# Post-Hoc Flow Preconditioning Experiments E0-E4

Runs the expanded shared protocol for E0 quadratic residual sanity, E1 fixed decoder inverse problem, E2 scalar graph-geometry landscapes, E3 nonsmooth residual map, and E4 tiny MLP weight-space optimization. The notebook is cache-first and delegates the heavy protocol implementation to helper modules.

In [ ]:
from __future__ import annotations

import json
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, Markdown, display

from post_train_research.loss_landscape_analysis.flow_preconditioning import ExperimentConfig, run_or_load
from post_train_research.loss_landscape_analysis.flow_preconditioning.reporting import geometry_summary, paired_deltas
from post_train_research.loss_landscape_analysis.flow_preconditioning.toy_mlp import MLPRegressionProblem, mlp_forward

plt.rcParams['figure.dpi'] = 130

cfg = ExperimentConfig(
    run_label='e0_e4_paperish',
    artifact_root='artifacts/loss_landscape_analysis/flow_preconditioning',
    cache_first=True,
    force_rerun=False,
    seeds=(0, 1, 2, 3, 4),
    k_tune=8,
    k_eval=32,
    budgets=(300, 1000),
    downstream_tuning_batch_size=288,
    downstream_eval_batch_size=288,
    parallel_contexts=5,
    e2_gamma_values=(0.3, 1.0, 3.0),
    e2_dims=(2, 4),
    e2_objectives=('rastrigin_abs', 'rosenbrock_abs'),
    e3_condition_numbers=(1.0, 1e2, 1e4),
    e4_rho_values=(0.0, 1e-3, 1e-2, 5e-2),
    e4_main_rho=1e-2,
)

tables = run_or_load(cfg)
figure_dir = tables.output_dir / 'figures'
print('output_dir:', tables.output_dir)
print('results:', tables.results.shape)
print('curves:', tables.curves.shape)
print('geometry:', tables.geometry.shape)
print('selected_lrs:', tables.selected_lrs.shape)
print('trajectories:', tables.trajectories.shape)
print('warped_grids:', tables.warped_grids.shape)


## Tables

In [ ]:
geom_summary = geometry_summary(tables.geometry)
delta_table = paired_deltas(tables.results)

display(Markdown('### Selected learning rates'))
display(tables.selected_lrs.sort_values(['experiment', 'task', 'condition_name', 'condition_value', 'seed', 'budget', 'method']).head(80))

display(Markdown('### Aggregate metrics'))
display(tables.aggregate.sort_values(['experiment', 'task', 'condition_name', 'condition_value', 'budget', 'optimizer', 'method']).head(120))

display(Markdown('### Geometry diagnostics'))
display(geom_summary.sort_values(['experiment', 'task', 'condition_name', 'condition_value', 'diagnostic', 'coordinate', 'seed']).head(120))

display(Markdown('### Paired AULC deltas'))
display(delta_table.groupby(['experiment', 'task', 'condition_name', 'condition_value', 'budget', 'optimizer', 'comparison'])['delta_aulc'].median().reset_index().head(120))


## Representative E0-E4 Plots

In [ ]:
def condition_filter(frame: pd.DataFrame, experiment: str, task: str, condition_name: str, condition_value: str):
    return frame[
        (frame['experiment'] == experiment)
        & (frame['task'] == task)
        & (frame['condition_name'] == condition_name)
        & (frame['condition_value'].astype(str) == str(condition_value))
    ]

def plot_condition(experiment: str, task: str, condition_name: str, condition_value: str, *, budget: int | None = None):
    budget = int(max(cfg.budgets) if budget is None else budget)
    curves = condition_filter(tables.curves, experiment, task, condition_name, condition_value)
    curves = curves[(curves['split'] == 'eval') & (curves['budget'] == budget)]
    if curves.empty:
        print('no curves for', experiment, task, condition_name, condition_value)
        return
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
    for method, frame in curves.groupby('method'):
        grouped = frame.groupby('step')['train_loss']
        median = grouped.median()
        q25 = grouped.quantile(0.25)
        q75 = grouped.quantile(0.75)
        axes[0].plot(median.index, median.values, label=method)
        axes[0].fill_between(median.index, q25.values, q75.values, alpha=0.12)
    axes[0].set_yscale('log')
    axes[0].set_title('median loss curves')
    axes[0].set_xlabel('step')
    axes[0].set_ylabel('loss')
    axes[0].grid(True, alpha=0.25)
    axes[0].legend(fontsize=7, ncol=2)

    deltas = condition_filter(delta_table, experiment, task, condition_name, condition_value)
    scatter = deltas[(deltas['budget'] == budget) & (deltas['comparison'] == 'trained_minus_direct')]
    for optimizer, frame in scatter.groupby('optimizer'):
        axes[1].scatter(frame['right_aulc'], frame['left_aulc'], s=20, alpha=0.7, label=optimizer)
    if not scatter.empty:
        lo = float(np.nanmin([scatter['right_aulc'].min(), scatter['left_aulc'].min()]))
        hi = float(np.nanmax([scatter['right_aulc'].max(), scatter['left_aulc'].max()]))
        axes[1].plot([lo, hi], [lo, hi], color='black', linewidth=1)
    axes[1].set_title('paired AULC')
    axes[1].set_xlabel('direct')
    axes[1].set_ylabel('trained-flow')
    axes[1].grid(True, alpha=0.25)
    axes[1].legend()

    geom = condition_filter(geom_summary, experiment, task, condition_name, condition_value)
    geom = geom[geom['diagnostic'] == 'pullback_metric']
    if not geom.empty:
        geom.boxplot(column='isometry_objective', by='coordinate', ax=axes[2])
    axes[2].set_title('heldout geometry R')
    axes[2].set_xlabel('coordinate')
    fig.suptitle(f'{experiment} / {task} / {condition_name}={condition_value} / T={budget}')
    plt.show()

representatives = [
    ('E0', 'quadratic_residual', 'cond_A', '1000.0'),
    ('E1', 'fixed_decoder_inverse', 'decoder', 'fixed'),
    ('E2', 'rastrigin_abs_d2', 'gamma', '1.0'),
    ('E3', 'relu_residual', 'cond_S', '100.0'),
    ('E4', 'tiny_mlp_weight_space', 'rho', str(cfg.e4_main_rho)),
]
for args in representatives:
    plot_condition(*args)


## Low-Dimensional Trajectories And Warped Grids

In [ ]:
def plot_low_dim_visuals(experiment: str, task: str, condition_name: str, condition_value: str):
    traj = condition_filter(tables.trajectories, experiment, task, condition_name, condition_value)
    warp = condition_filter(tables.warped_grids, experiment, task, condition_name, condition_value)
    fig, axes = plt.subplots(1, 2, figsize=(11, 5), constrained_layout=True)
    if not traj.empty:
        for (method, start_index), frame in traj.groupby(['method', 'start_index']):
            points = np.array([json.loads(value) for value in frame.sort_values('step')['z_json']], dtype=float)
            if points.shape[1] >= 2:
                axes[0].plot(points[:, 0], points[:, 1], linewidth=1.2, label=f'{method}/s{start_index}')
                axes[0].scatter(points[0, 0], points[0, 1], s=14, color='black')
    axes[0].set_title('trajectories in original z-space')
    axes[0].set_xlabel('z0')
    axes[0].set_ylabel('z1')
    axes[0].grid(True, alpha=0.25)
    axes[0].legend(fontsize=6, ncol=2)

    warp = warp[warp['flow'] == 'trained_flow']
    if not warp.empty:
        seed = sorted(warp['seed'].unique())[0]
        warp = warp[warp['seed'] == seed]
        for (_axis, _line_index), line in warp.groupby(['axis', 'line_index']):
            line = line.sort_values('point_index')
            axes[1].plot(line['u0'], line['u1'], color='tab:blue', linewidth=0.8, alpha=0.7)
    axes[1].set_title('warped grid u=i(z)')
    axes[1].set_xlabel('u0')
    axes[1].set_ylabel('u1')
    axes[1].grid(True, alpha=0.25)
    fig.suptitle(f'{experiment} / {task} / {condition_name}={condition_value}')
    plt.show()

plot_low_dim_visuals('E1', 'fixed_decoder_inverse', 'decoder', 'fixed')
plot_low_dim_visuals('E2', 'rastrigin_abs_d2', 'gamma', '1.0')


## E4 Representative Predictions

In [ ]:
e4 = condition_filter(tables.results, 'E4', 'tiny_mlp_weight_space', 'rho', str(cfg.e4_main_rho))
e4 = e4[(e4['split'] == 'eval') & (~e4['candidate']) & (e4['budget'] == int(max(cfg.budgets)))]
pairs = e4[e4['method'].isin(['direct_adam', 'trained_flow_adam'])]
chosen = None
for key, frame in pairs.groupby(['seed', 'start_index']):
    if {'direct_adam', 'trained_flow_adam'}.issubset(set(frame['method'])):
        chosen = key
        break
if chosen is None:
    raise RuntimeError('No matched E4 direct/trained Adam pair found')
seed, start_index = chosen
plot_cfg = replace(cfg, device='cpu', dtype='float32')
problem = MLPRegressionProblem(plot_cfg, seed=int(seed), device=torch.device('cpu'), dtype=torch.float32)
x = problem.x_test.detach().cpu()
target = problem.y_test.detach().cpu()
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x.numpy(), target.numpy(), color='black', linewidth=2, label='target')
for method, frame in pairs[(pairs['seed'] == seed) & (pairs['start_index'] == start_index)].groupby('method'):
    theta = torch.tensor(json.loads(frame.iloc[0]['final_theta_json']), dtype=torch.float32)
    pred = mlp_forward(theta, x).detach().cpu()
    ax.plot(x.numpy(), pred.numpy(), label=f'{method} final')
ax.set_title(f'E4 representative predictions: seed={seed}, start={start_index}')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.25)
ax.legend()
fig.savefig(figure_dir / 'E4_representative_predictions.png', dpi=160, bbox_inches='tight')
plt.show()


## Saved Figures

In [ ]:
print('figure_dir:', figure_dir)
for name, path in list(tables.figure_paths.items())[:20]:
    print(name, '->', path)


## Automatic Interpretation

In [ ]:
display(Markdown(tables.interpretation_markdown))
print('interpretation saved to:', tables.output_dir / 'interpretation.md')
